## Generate control plots

This example will demonstrate how one method for creating control plots surrounding an area of interest to use for analyses like carbon baselining. This example creates a set number of control plots within a buffer zone around an AOI, ensuring that the overall area of the control plots meets some threshold of the overall bounding box area (e.g. total control plot area is more than 1% of the bounding box area).

Variations on this process could include stratified placement of the control plots to ensure a variety of habitats are represented, but this example demonstrates the most basic example only focusing on meeting an overall coverage threshold.

The same approach can be applied to a number of different use cases and can be used on any subscription you have created. Just use one of your own subscription ids and swap out variable names as necessary. 

First we set up the Cecil client and import the required packages.

In [1]:
import cecil
import geopandas as gpd
import numpy as np
from shapely.ops import unary_union
from shapely.geometry import Point, box, shape, mapping, MultiPolygon
import matplotlib.pyplot as plt

client = cecil.Client()

### Set up control plot configuration

Select a main AOI around which to position the control plots. In this example, the AOI has already been added to the Cecil platform.

In [2]:
main_aoi = client.get_aoi('90d86599-9157-4e46-a817-35fec152d567')

Set up some configuration parameters for the control plots, such as the minimum coverage threshold, and the desired number of control plots. 

In [3]:
# Configuration parameters for control plots.
BUFFER_DISTANCE_M = 10000 # Distance outside of main plot to select control plots  
CONTROL_PLOT_NUMBER = 30 # Number of control plots to generate
CONTROL_PLOT_SIZE_M = 500 # Size of each control plot in meters (e.g. 500mx500m)
MIN_COVERAGE_PCT = 1.0  # Set the minimum coverage percentage of the buffered area
RANDOM_SEED = 42 # Random seed for reproducibility (if desired) 

### Calculate the necessary number of plots to meet coverage threshold
In this example, the area of the plots is held constant while the number of plots can be dynamically adjusted to ensure the coverage threshold is met.

In [4]:
# Get the AOI polygon and convert to UTM (meters).
aoi_polygon = shape(main_aoi.geometry)
aoi = gpd.GeoDataFrame(geometry=[aoi_polygon], crs="EPSG:4326")
aoi_utm = aoi.estimate_utm_crs()
aoi = aoi.to_crs(aoi_utm)

# Create buffer around AOI (within which to place control plots).
aoi_buffer = aoi.buffer(BUFFER_DISTANCE_M)

# Get the area of the buffer area bounding box.
bounds = aoi_buffer.total_bounds 
bbox = box(*bounds)
bbox_area = bbox.area  # Area in square meters

# Calculate plot area and number of plots needed to meet coverage threshold.
plot_area = CONTROL_PLOT_SIZE_M ** 2  # Square meters
min_total_area = bbox_area * (MIN_COVERAGE_PCT / 100)
n_plots_min = int(np.ceil(min_total_area / plot_area))
n_plots = max(CONTROL_PLOT_NUMBER, n_plots_min)

print(f"Generating {n_plots} control plots to meet minimum coverage of {MIN_COVERAGE_PCT}%")

Generating 30 control plots to meet minimum coverage of 1.0%


### Generate the control plots according to configuration parameters

In [5]:
def generate_control_plots(random_seed, n_plots, aoi, aoi_buffer, control_plot_size_m, max_attempts=100):
    """
    Generate random control plots within the buffer area, checking to ensure the 
    control plots don't overlap with the main AOI or each other.
    """
    # Generate random control plots within the buffer.
    np.random.seed(random_seed)
    control_plots = []
    attempts = 0

    # Get the AOI geometry for checking overlap.
    aoi_geom = aoi.geometry.iloc[0]
    buffer_geom = aoi_buffer.geometry.iloc[0]

    # Loop until we have enough control plots or reach max attempts.
    while len(control_plots) < n_plots and attempts < max_attempts:
        attempts += 1
        
        # Generate random center point within bounding box.
        x = np.random.uniform(bounds[0], bounds[2])
        y = np.random.uniform(bounds[1], bounds[3])
        center = Point(x, y)
        
        # Create square plot around center.
        half_size = control_plot_size_m / 2
        plot = box(x - half_size, y - half_size, x + half_size, y + half_size)
        
        # Check if plot is within buffer, doesn't overlap with AOI, and doesn't overlap with existing plots
        if (buffer_geom.contains(plot) and 
            not aoi_geom.contains(center) and
            not any(plot.intersects(existing) for existing in control_plots)):
            control_plots.append(plot)

    # Create GeoDataFrame with control plots.
    control_gdf = gpd.GeoDataFrame(
        {'plot_id': range(len(control_plots))},
        geometry=control_plots,
        crs=aoi_utm
    )
    return control_gdf, attempts

In [6]:
control_gdf, attempts = generate_control_plots(RANDOM_SEED, n_plots, aoi, aoi_buffer, CONTROL_PLOT_SIZE_M, max_attempts=100)
print(f"\nGenerated {len(control_gdf)} control plots in {attempts} attempts")


Generated 30 control plots in 42 attempts


Check true coverage of the control plots over the bounding box.

In [7]:
# Get the bounding box of the control plots.
control_bounds = control_gdf.total_bounds 
control_bbox = box(*control_bounds)
control_bbox_area = control_bbox.area

# Calculate total area of control plots.
control_total_area = control_gdf.geometry.area.sum()

print(f"Actual coverage: {(control_total_area / control_bbox_area) * 100:.2f}%")

Actual coverage: 2.11%


### Visualise control plots

Visualise the main AOI, buffer area, and control plots over satellite imagery basemap.

In [8]:
import folium

# Convert all geometries to WGS84 for mapping
aoi_wgs84 = aoi.to_crs("EPSG:4326")
buffer_wgs84 = aoi_buffer.to_crs("EPSG:4326")
control_gdf_wgs84 = control_gdf.to_crs("EPSG:4326")

# Get center of AOI for map
center_lat = aoi_wgs84.geometry.iloc[0].centroid.y
center_lon = aoi_wgs84.geometry.iloc[0].centroid.x

# Create map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Add satellite imagery as alternative basemap
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite',
    overlay=False,
    control=True
).add_to(m)

# Add buffer boundary
folium.GeoJson(
    buffer_wgs84.boundary,
    name='Buffer boundary',
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0
    }
).add_to(m)

# Add AOI
folium.GeoJson(
    aoi_wgs84,
    name='AOI',
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'darkred',
        'weight': 2,
        'fillOpacity': 0.6
    }
).add_to(m)

# Add control plots
folium.GeoJson(
    control_gdf_wgs84,
    name='Control plots',
    style_function=lambda x: {
        'fillColor': 'green',
        'color': 'darkgreen',
        'weight': 1,
        'fillOpacity': 0.5
    },
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'], aliases=['Plot ID:'])
).add_to(m)


folium.LayerControl().add_to(m)

m

### Add AOIs to Cecil platform

To add the control plots to the Cecil platform, they can either be added individually as unique AOIs, or they can be added as a single MultiPolygon AOI. The right decision will depend on the exact use case, but both options are demonstrated below.

**Add control plots as unique AOIs:**

In [9]:
# Ensure control plots are in WGS84 for creating Cecil AOIs.
control_gdf_wgs84 = control_gdf.to_crs("EPSG:4326")

# To keep track of created AOIs, group with a project code.
project_code = "project123"

# Loop through the control plots and create AOIs in Cecil.
for idx, row in control_gdf_wgs84.iterrows():
    geom = row.geometry
    geojson_geom = mapping(geom)

    # Create AOI in Cecil.
    aoi = client.create_aoi(
        external_ref=f"{project_code}-plot_{row['plot_id']}",
        geometry=geojson_geom
    )
    print(f"Created AOI with ID: {aoi.id} and external_ref: {aoi.external_ref}")

Created AOI with ID: 560d2f7a-6d50-48ce-8354-c66b32fcf081 and external_ref: project123-plot_0
Created AOI with ID: 945201c7-936a-47ec-a23a-24835065f1bf and external_ref: project123-plot_1
Created AOI with ID: 91b0d394-f943-49b4-86c6-631b0c6c75f8 and external_ref: project123-plot_2
Created AOI with ID: d5e92aa2-f99a-4282-87b8-510548eb7d92 and external_ref: project123-plot_3
Created AOI with ID: 0fcbfc72-b47f-402b-bee3-edb822faf75c and external_ref: project123-plot_4
Created AOI with ID: 03ee5197-5c82-4c6d-82fc-9701d82b5d1a and external_ref: project123-plot_5
Created AOI with ID: 4aefa482-c167-4f22-bed0-ad344461bc32 and external_ref: project123-plot_6
Created AOI with ID: 55d5cf27-a949-4468-a98d-dbb6715b191b and external_ref: project123-plot_7
Created AOI with ID: 03fa9d52-d2b0-4e74-8af1-1abd82625394 and external_ref: project123-plot_8
Created AOI with ID: dcafc7a6-d609-4855-8743-45c6f4299094 and external_ref: project123-plot_9
Created AOI with ID: f31efb58-6d9f-482c-8882-6d783dc107d8 an

Visualise with folium:

In [10]:
# Get the geometry of the control plots we created.
project_plots = [aoi for aoi in client.list_aois() if aoi.external_ref.startswith(project_code)]
project_plot_geoms = [client.get_aoi(aoi.id).geometry for aoi in project_plots]

# Calculate the center of the control plots for map centering.
shapely_geoms = [shape(geom) for geom in project_plot_geoms]
center_geom = unary_union(shapely_geoms)

m = folium.Map(location=[center_geom.centroid.y, center_geom.centroid.x], zoom_start=11)
for geom in project_plot_geoms:
    folium.GeoJson(geom).add_to(m)
m

**Add control plots as a single MultiPolygon:**

In [11]:
# Ensure control plots are in WGS84 for creating Cecil AOIs.
control_gdf_wgs84 = control_gdf.to_crs("EPSG:4326")

# Create a MultiPolygon from all control plot geometries.
multi_polygon = MultiPolygon(control_gdf_wgs84.geometry.tolist())

# Convert to GeoJSON.
geojson_geom = mapping(multi_polygon)

# Create AOI in Cecil.
multi_aoi = client.create_aoi(
    external_ref="control_plots_multi",
    geometry=geojson_geom
)

print(f"Created AOI with ID: {multi_aoi.id} and external_ref: {multi_aoi.external_ref}")


Created AOI with ID: a57b9873-10dd-4e40-a231-83f041ac17c8 and external_ref: control_plots_multi


Visualise with folium:

In [12]:
multi_aoi_geom = shape(client.get_aoi(multi_aoi.id).geometry)
m = folium.Map(location=[multi_aoi_geom.centroid.y, multi_aoi_geom.centroid.x], zoom_start=11)
folium.GeoJson(multi_aoi.geometry).add_to(m)
m